# 第 2 章　指标记法与符号系统：计算伴侣

本 Notebook 是《张量分析》第 2 章的配套电子资源。它不复制纸质教材的完整叙述，而是围绕本章最适合计算实验的内容建立“**指标审计—显式循环—数值验证—结构解释**”闭环。

本章采用三维右手笛卡尔坐标系，拉丁指标 $i,j,k,\ldots$ 均取 $1,2,3$。

> **代码约定**
>
> - 数学指标 $1,2,3$ 对应 Python 下标 `0,1,2`；
> - 自由指标通常对应输出数组的外层循环；
> - 哑指标（求和指标）通常对应内部求和循环；
> - 涉及指标缩并时优先使用显式 `for` 循环；
> - 不使用 `@` 和 `np.einsum`；
> - NumPy / SymPy 主要用于数组存储、独立核验与符号化简。

本 Notebook 的数学公式统一使用 `$...$` 与 `$$...$$`，以兼容 Jupyter Book 2 / MyST。

## 学习目标

完成本 Notebook 后，应能够：

1. 把自由指标与哑指标翻译成程序中的外层与内层循环；
2. 用显式循环实现矩阵—矢量乘积、矩阵乘法和多重缩并；
3. 数值区分 $A_{ij}A_{ij}$ 与 $A_{ij}A_{ji}$；
4. 构造并使用克罗内克符号 $\delta_{ij}$；
5. 构造列维–奇维塔符号 $\varepsilon_{ijk}$ 并实现叉积；
6. 验证 $\varepsilon$–$\delta$ 恒等式及拉格朗日恒等式；
7. 实现四阶张量缩并 $\sigma_{ij}=C_{ijkl}\varepsilon_{kl}$；
8. 构造叉乘矩阵 $[\boldsymbol\omega]_\times$；
9. 设计对称张量六分量压缩存储；
10. 用符号计算核对对流加速度；
11. 理解“指标计数”能够发现哪些语法错误，又不能发现哪些语义错误。

In [1]:
import re
import numpy as np
import sympy as sp

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(20260823)

print("NumPy version:", np.__version__)
print("SymPy version:", sp.__version__)

NumPy version: 2.3.5
SymPy version: 1.14.0


## 1. 自由指标与哑指标：从公式到循环

考虑

$$
y_i=A_{ij}u_j.
$$

$i$ 是自由指标，决定输出 $y_i$ 的位置；$j$ 是哑指标，负责内部求和。因此程序结构自然是

```text
for i:       # 自由指标
    for j:   # 哑指标
        累加到 y[i]
```

先用一个小整数例子验证。

In [2]:
A = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
])
u = np.array([2.0, -1.0, 3.0])

y = np.zeros(3)

for i in range(3):          # 自由指标 i：决定输出位置
    for j in range(3):      # 哑指标 j：内部求和
        y[i] += A[i, j] * u[j]

print("y_i = A_ij u_j =", y)

# NumPy 仅用于独立核验
y_check = np.dot(A, u)
print("NumPy check =", y_check)

assert np.allclose(y, y_check)

y_i = A_ij u_j = [ 9. 21. 33.]
NumPy check = [ 9. 21. 33.]


### 一个最小指标审计器：只看“出现次数”

下列函数不解析完整 LaTeX，只接受“每个因子的指标列表”，用于展示最基本的指标计数逻辑：

- 出现一次：自由指标；
- 出现两次：哑指标；
- 出现三次或更多：在本书标准爱因斯坦约定下非法。

In [3]:
from collections import Counter

def audit_indices(*index_groups):
    counts = Counter()
    for group in index_groups:
        counts.update(group)

    free = sorted([idx for idx, n in counts.items() if n == 1])
    dummy = sorted([idx for idx, n in counts.items() if n == 2])
    illegal = {idx: n for idx, n in counts.items() if n >= 3}

    return {
        "counts": dict(counts),
        "free": free,
        "dummy": dummy,
        "illegal": illegal,
    }

examples = {
    "A_ij u_j": (("i", "j"), ("j",)),
    "A_ij B_jk": (("i", "j"), ("j", "k")),
    "A_ij B_ij": (("i", "j"), ("i", "j")),
    "T_ijk u_j u_k": (("i", "j", "k"), ("j",), ("k",)),
    "A_ii x_i": (("i", "i"), ("i",)),
}

for name, groups in examples.items():
    print(name, "->", audit_indices(*groups))

A_ij u_j -> {'counts': {'i': 1, 'j': 2}, 'free': ['i'], 'dummy': ['j'], 'illegal': {}}
A_ij B_jk -> {'counts': {'i': 1, 'j': 2, 'k': 1}, 'free': ['i', 'k'], 'dummy': ['j'], 'illegal': {}}
A_ij B_ij -> {'counts': {'i': 2, 'j': 2}, 'free': [], 'dummy': ['i', 'j'], 'illegal': {}}
T_ijk u_j u_k -> {'counts': {'i': 1, 'j': 2, 'k': 2}, 'free': ['i'], 'dummy': ['j', 'k'], 'illegal': {}}
A_ii x_i -> {'counts': {'i': 3}, 'free': [], 'dummy': [], 'illegal': {'i': 3}}


这个审计器能发现“某指标出现三次”等形式错误，但还不能判断微分算子的作用范围、张量真实阶次或公式的物理含义。后面会再构造一个稍完整的教学型语法检查器。

## 2. 数字赋能 02-A：显式循环验证求和约定

### 2.1 小整数矩阵乘法

$$
C_{ik}=A_{ij}B_{jk}.
$$

$i,k$ 是自由指标，$j$ 是哑指标。固定 $i=1,k=2$ 时，

$$
C_{12}=A_{11}B_{12}+A_{12}B_{22}.
$$

In [4]:
A_small = np.array([
    [1, 2],
    [3, 4],
])
B_small = np.array([
    [5, 6],
    [7, 8],
])

C_small = np.zeros((2, 2), dtype=int)

for i in range(2):          # 自由指标 i
    for k in range(2):      # 自由指标 k
        for j in range(2):  # 哑指标 j
            C_small[i, k] += A_small[i, j] * B_small[j, k]

print("C_small =")
print(C_small)
print("C_12 =", C_small[0, 1])

assert C_small[0, 1] == 22
assert np.array_equal(C_small, np.array([[19, 22], [43, 50]]))

C_small =
[[19 22]
 [43 50]]
C_12 = 22


### 2.2 四类典型输出

下面同时实现

$$
y_i=A_{ij}u_j,
\qquad
C_{ik}=A_{ij}B_{jk},
$$

$$
s=A_{ij}B_{ij},
\qquad
z_i=T_{ijk}u_ju_k.
$$

In [5]:
A = rng.random((3, 3))
B = rng.random((3, 3))
u = rng.random(3)
T = rng.random((3, 3, 3))

y = np.zeros(3)
for i in range(3):
    for j in range(3):
        y[i] += A[i, j] * u[j]

C = np.zeros((3, 3))
for i in range(3):
    for k in range(3):
        for j in range(3):
            C[i, k] += A[i, j] * B[j, k]

s = 0.0
for i in range(3):
    for j in range(3):
        s += A[i, j] * B[i, j]

z = np.zeros(3)
for i in range(3):
    for j in range(3):
        for k in range(3):
            z[i] += T[i, j, k] * u[j] * u[k]

print("A_ij u_j shape:", y.shape)
print("A_ij B_jk shape:", C.shape)
print("A_ij B_ij shape:", np.shape(s))
print("T_ijk u_j u_k shape:", z.shape)

assert y.shape == (3,)
assert C.shape == (3, 3)
assert np.shape(s) == ()
assert z.shape == (3,)

A_ij u_j shape: (3,)
A_ij B_jk shape: (3, 3)
A_ij B_ij shape: ()
T_ijk u_j u_k shape: (3,)


### 2.3 矩阵乘法不交换，双点积交换输入不变

一般有

$$
\mathbf A\mathbf B\ne\mathbf B\mathbf A,
$$

而

$$
A_{ij}B_{ij}=B_{ij}A_{ij}.
$$

In [6]:
BA = np.zeros((3, 3))
for i in range(3):
    for k in range(3):
        for j in range(3):
            BA[i, k] += B[i, j] * A[j, k]

double_dot_AB = 0.0
double_dot_BA = 0.0

for i in range(3):
    for j in range(3):
        double_dot_AB += A[i, j] * B[i, j]
        double_dot_BA += B[i, j] * A[i, j]

print("AB == BA ?", np.allclose(C, BA))
print("A:B =", double_dot_AB)
print("B:A =", double_dot_BA)

assert np.isclose(double_dot_AB, double_dot_BA)

AB == BA ? False
A:B = 1.7242306288366736
B:A = 1.7242306288366736


## 3. 两种容易混淆的缩并

$$
A_{ij}A_{ij}
=
\|\mathbf A\|_F^2,
$$

而

$$
A_{ij}A_{ji}
=
\operatorname{tr}(\mathbf A^2).
$$

二者一般不同。

In [7]:
A_test = np.array([
    [0.0,  1.0, 0.0],
    [-2.0, 0.0, 0.0],
    [0.0,  0.0, 3.0],
])

aij_aij = 0.0
aij_aji = 0.0

for i in range(3):
    for j in range(3):
        aij_aij += A_test[i, j] * A_test[i, j]
        aij_aji += A_test[i, j] * A_test[j, i]

print("A_ij A_ij =", aij_aij)
print("A_ij A_ji =", aij_aji)

assert np.isclose(aij_aij, 14.0)
assert np.isclose(aij_aji, 5.0)

A_ij A_ij = 14.0
A_ij A_ji = 5.0


### 对称—反对称分解

$$
A_{ij}=S_{ij}+W_{ij},
\qquad
S_{ij}=S_{ji},
\qquad
W_{ij}=-W_{ji}.
$$

验证

$$
S_{ij}W_{ij}=0,
$$

$$
A_{ij}A_{ij}
=
\|\mathbf S\|_F^2+\|\mathbf W\|_F^2,
$$

$$
A_{ij}A_{ji}
=
\|\mathbf S\|_F^2-\|\mathbf W\|_F^2.
$$

In [8]:
S = 0.5 * (A_test + A_test.T)
W = 0.5 * (A_test - A_test.T)

S_dot_W = 0.0
S_norm_sq = 0.0
W_norm_sq = 0.0

for i in range(3):
    for j in range(3):
        S_dot_W += S[i, j] * W[i, j]
        S_norm_sq += S[i, j] * S[i, j]
        W_norm_sq += W[i, j] * W[i, j]

print("S:W =", S_dot_W)
print("||S||_F^2 + ||W||_F^2 =", S_norm_sq + W_norm_sq)
print("||S||_F^2 - ||W||_F^2 =", S_norm_sq - W_norm_sq)

assert np.isclose(S_dot_W, 0.0)
assert np.isclose(aij_aij, S_norm_sq + W_norm_sq)
assert np.isclose(aij_aji, S_norm_sq - W_norm_sq)

S:W = 0.0
||S||_F^2 + ||W||_F^2 = 14.0
||S||_F^2 - ||W||_F^2 = 5.0


## 4. 数字赋能 02-B：克罗内克符号 $\delta_{ij}$

验证

$$
\delta_{ii}=3,
\qquad
\delta_{ij}u_j=u_i,
$$

$$
\delta_{im}A_{mj}=A_{ij},
\qquad
\delta_{ij}A_{ij}=\operatorname{tr}\mathbf A.
$$

In [9]:
delta = np.eye(3)

delta_ii = 0.0
for i in range(3):
    delta_ii += delta[i, i]

u = rng.random(3)
u_screened = np.zeros(3)
for i in range(3):
    for j in range(3):
        u_screened[i] += delta[i, j] * u[j]

A = rng.random((3, 3))
A_screened = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        for m in range(3):
            A_screened[i, j] += delta[i, m] * A[m, j]

trace_by_delta = 0.0
for i in range(3):
    for j in range(3):
        trace_by_delta += delta[i, j] * A[i, j]

trace_direct = 0.0
for i in range(3):
    trace_direct += A[i, i]

print("delta_ii =", delta_ii)
print("delta_ij u_j =", u_screened)
print("trace by delta =", trace_by_delta)

assert np.isclose(delta_ii, 3.0)
assert np.allclose(u_screened, u)
assert np.allclose(A_screened, A)
assert np.isclose(trace_by_delta, trace_direct)

delta_ii = 3.0
delta_ij u_j = [0.76348  0.405111 0.781547]
trace by delta = 1.795590845980171


## 5. 数字赋能 02-C：构造 $\varepsilon_{ijk}$ 并验证叉积

$$
w_i=\varepsilon_{ijk}u_jv_k.
$$

In [10]:
epsilon = np.zeros((3, 3, 3), dtype=int)

epsilon[0, 1, 2] = 1
epsilon[1, 2, 0] = 1
epsilon[2, 0, 1] = 1

epsilon[0, 2, 1] = -1
epsilon[2, 1, 0] = -1
epsilon[1, 0, 2] = -1

print("epsilon_123 =", epsilon[0, 1, 2])
print("epsilon_231 =", epsilon[1, 2, 0])
print("epsilon_232 =", epsilon[1, 2, 1])

assert epsilon[0, 1, 2] == 1
assert epsilon[1, 2, 0] == 1
assert epsilon[1, 2, 1] == 0

epsilon_123 = 1
epsilon_231 = 1
epsilon_232 = 0


In [11]:
def cross_by_indices(left, right, eps=epsilon):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    result = np.zeros(3)

    for i in range(3):
        for j in range(3):
            for k in range(3):
                result[i] += eps[i, j, k] * left[j] * right[k]

    return result

u = np.array([3.0, -2.0, 1.0])
v = np.array([-1.0, 4.0, 5.0])

w_loop = cross_by_indices(u, v)
w_check = np.cross(u, v)

print("显式指标循环:", w_loop)
print("np.cross 核验:", w_check)
print("u × u =", cross_by_indices(u, u))
print("v × u =", cross_by_indices(v, u))

assert np.allclose(w_loop, w_check)
assert np.allclose(cross_by_indices(u, u), 0.0)
assert np.allclose(cross_by_indices(v, u), -w_loop)

显式指标循环: [-14. -16.  10.]
np.cross 核验: [-14. -16.  10.]
u × u = [0. 0. 0.]
v × u = [ 14.  16. -10.]


## 6. $\varepsilon$–$\delta$ 恒等式：逐项验证

$$
\varepsilon_{ijk}\varepsilon_{\ell jk}
=
2\delta_{i\ell},
$$

以及

$$
\varepsilon_{ijk}\varepsilon_{ijk}=6.
$$

In [12]:
lhs_two_contract = np.zeros((3, 3))

for i in range(3):
    for ell in range(3):
        for j in range(3):
            for k in range(3):
                lhs_two_contract[i, ell] += (
                    epsilon[i, j, k] * epsilon[ell, j, k]
                )

rhs_two_contract = 2.0 * delta
residual_matrix = lhs_two_contract - rhs_two_contract

triple_contract = 0.0
for i in range(3):
    for j in range(3):
        for k in range(3):
            triple_contract += epsilon[i, j, k] * epsilon[i, j, k]

print("epsilon_ijk epsilon_ljk =")
print(lhs_two_contract)
print("residual =")
print(residual_matrix)
print("epsilon_ijk epsilon_ijk =", triple_contract)

assert np.allclose(residual_matrix, 0.0)
assert np.isclose(triple_contract, 6.0)

epsilon_ijk epsilon_ljk =
[[2. 0. 0.]
 [0. 2. 0.]
 [0. 0. 2.]]
residual =
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
epsilon_ijk epsilon_ijk = 6.0


再验证更完整的单指标缩并

$$
\varepsilon_{ijk}\varepsilon_{i\ell m}
=
\delta_{j\ell}\delta_{km}
-
\delta_{jm}\delta_{k\ell}.
$$

这里留下 $j,k,\ell,m$ 四个自由指标，需要逐项检查 $3^4=81$ 个分量。

In [13]:
lhs_one_contract = np.zeros((3, 3, 3, 3))
rhs_one_contract = np.zeros((3, 3, 3, 3))

for j in range(3):
    for k in range(3):
        for ell in range(3):
            for m in range(3):
                for i in range(3):
                    lhs_one_contract[j, k, ell, m] += (
                        epsilon[i, j, k] * epsilon[i, ell, m]
                    )

                rhs_one_contract[j, k, ell, m] = (
                    delta[j, ell] * delta[k, m]
                    - delta[j, m] * delta[k, ell]
                )

one_contract_residual = lhs_one_contract - rhs_one_contract

print("max residual =", np.max(np.abs(one_contract_residual)))
assert np.allclose(one_contract_residual, 0.0)

max residual = 0.0


### 故意翻转一个排列符号：测试敏感性

把 $\varepsilon_{123}$ 从 $+1$ 改为 $-1$。此时：

- 完全平方缩并仍可能给出 $6$；
- 双指标缩并仍可能通过；
- 单指标缩并恒等式会产生非零残差。

这说明测试是否能发现错误，取决于所选择的不变量或恒等式是否足够敏感。

In [14]:
epsilon_bad = epsilon.copy()
epsilon_bad[0, 1, 2] = -1

bad_triple = 0.0
for i in range(3):
    for j in range(3):
        for k in range(3):
            bad_triple += epsilon_bad[i, j, k] * epsilon_bad[i, j, k]

bad_two = np.zeros((3, 3))
for i in range(3):
    for ell in range(3):
        for j in range(3):
            for k in range(3):
                bad_two[i, ell] += (
                    epsilon_bad[i, j, k] * epsilon_bad[ell, j, k]
                )

bad_one = np.zeros((3, 3, 3, 3))
for j in range(3):
    for k in range(3):
        for ell in range(3):
            for m in range(3):
                for i in range(3):
                    bad_one[j, k, ell, m] += (
                        epsilon_bad[i, j, k] * epsilon_bad[i, ell, m]
                    )

bad_one_residual = bad_one - rhs_one_contract

print("modified triple contraction =", bad_triple)
print(
    "max residual of double contraction =",
    np.max(np.abs(bad_two - 2.0 * delta))
)
print(
    "max residual of one-index contraction =",
    np.max(np.abs(bad_one_residual))
)

assert np.isclose(bad_triple, 6.0)
assert np.allclose(bad_two, 2.0 * delta)
assert not np.allclose(bad_one_residual, 0.0)

modified triple contraction = 6.0
max residual of double contraction = 0.0
max residual of one-index contraction = 2.0


## 7. 数字赋能 02-D：SymPy 验证拉格朗日恒等式

$$
(\mathbf A\times\mathbf B)\cdot(\mathbf C\times\mathbf D)
=
(\mathbf A\cdot\mathbf C)(\mathbf B\cdot\mathbf D)
-
(\mathbf A\cdot\mathbf D)(\mathbf B\cdot\mathbf C).
$$

左端为

$$
\varepsilon_{ijk}\varepsilon_{i\ell m}
A_jB_kC_\ell D_m.
$$

In [15]:
A_sym = sp.symbols("A1:4")
B_sym = sp.symbols("B1:4")
C_sym = sp.symbols("C1:4")
D_sym = sp.symbols("D1:4")

def epsilon_sum(Avec, Bvec, Cvec, Dvec):
    result = 0
    for i in range(3):
        for j in range(3):
            for k in range(3):
                for ell in range(3):
                    for m in range(3):
                        result += (
                            sp.LeviCivita(i, j, k)
                            * sp.LeviCivita(i, ell, m)
                            * Avec[j] * Bvec[k]
                            * Cvec[ell] * Dvec[m]
                        )
    return sp.expand(result)

lhs = epsilon_sum(A_sym, B_sym, C_sym, D_sym)

rhs = (
    sum(A_sym[q] * C_sym[q] for q in range(3))
    * sum(B_sym[q] * D_sym[q] for q in range(3))
    - sum(A_sym[q] * D_sym[q] for q in range(3))
    * sum(B_sym[q] * C_sym[q] for q in range(3))
)

residual = sp.simplify(lhs - rhs)
swap_residual = sp.simplify(
    epsilon_sum(B_sym, A_sym, C_sym, D_sym) + lhs
)

print("LHS - RHS =", residual)
print("交换 A、B 后与原结果之和 =", swap_residual)

assert residual == 0
assert swap_residual == 0

LHS - RHS = 0
交换 A、B 后与原结果之和 = 0


## 8. 工程应用：航天器姿态动力学中的陀螺项

$$
H_k=I_{k\ell}\omega_\ell,
$$

$$
M_i^{(\mathrm{gyro})}
=
\varepsilon_{ijk}\omega_jH_k.
$$

在主惯性轴坐标系中验证三个分量公式。

In [16]:
I1, I2, I3 = 2.0, 3.0, 5.0
omega = np.array([1.2, -0.7, 0.9])

I_matrix = np.diag([I1, I2, I3])

H = np.zeros(3)
for k in range(3):
    for ell in range(3):
        H[k] += I_matrix[k, ell] * omega[ell]

M = np.zeros(3)
for i in range(3):
    for j in range(3):
        for k in range(3):
            M[i] += epsilon[i, j, k] * omega[j] * H[k]

M_formula = np.array([
    (I3 - I2) * omega[1] * omega[2],
    (I1 - I3) * omega[2] * omega[0],
    (I2 - I1) * omega[0] * omega[1],
])

print("H =", H)
print("M from indices =", M)
print("M from component formulas =", M_formula)

assert np.allclose(M, M_formula)

H = [ 2.4 -2.1  4.5]
M from indices = [-1.26 -3.24 -0.84]
M from component formulas = [-1.26 -3.24 -0.84]


## 9. 习题 2.10：四阶张量缩并

$$
\sigma_{ij}
=
C_{ijkl}\varepsilon_{kl}.
$$

采用各向同性线弹性张量

$$
C_{ijkl}
=
\lambda\delta_{ij}\delta_{kl}
+
\mu(
\delta_{ik}\delta_{j\ell}
+
\delta_{i\ell}\delta_{jk}
),
$$

应得到

$$
\sigma_{ij}
=
\lambda\varepsilon_{kk}\delta_{ij}
+
2\mu\varepsilon_{ij}.
$$

In [17]:
lam = 40.0
mu = 30.0

C4 = np.zeros((3, 3, 3, 3))

for i in range(3):
    for j in range(3):
        for k in range(3):
            for ell in range(3):
                C4[i, j, k, ell] = (
                    lam * delta[i, j] * delta[k, ell]
                    + mu * (
                        delta[i, k] * delta[j, ell]
                        + delta[i, ell] * delta[j, k]
                    )
                )

strain = np.array([
    [ 0.010,  0.002, -0.001],
    [ 0.002, -0.004,  0.003],
    [-0.001,  0.003,  0.006],
])

stress_loop = np.zeros((3, 3))

for i in range(3):
    for j in range(3):
        for k in range(3):
            for ell in range(3):
                stress_loop[i, j] += C4[i, j, k, ell] * strain[k, ell]

trace_strain = 0.0
for k in range(3):
    trace_strain += strain[k, k]

stress_exact = lam * trace_strain * delta + 2.0 * mu * strain

print("stress from four loops =")
print(stress_loop)
print("analytic stress =")
print(stress_exact)

assert np.allclose(stress_loop, stress_exact)
assert np.allclose(stress_loop, stress_loop.T)

stress from four loops =
[[ 1.08  0.12 -0.06]
 [ 0.12  0.24  0.18]
 [-0.06  0.18  0.84]]
analytic stress =
[[ 1.08  0.12 -0.06]
 [ 0.12  0.24  0.18]
 [-0.06  0.18  0.84]]


若 $C_{ijkl}$ 不具有所需材料对称性，即使输入应变对称，输出应力也不一定对称。

In [18]:
C_bad = rng.random((3, 3, 3, 3))

stress_bad = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        for k in range(3):
            for ell in range(3):
                stress_bad[i, j] += C_bad[i, j, k, ell] * strain[k, ell]

asymmetry = np.linalg.norm(stress_bad - stress_bad.T)

print("||sigma - sigma^T|| =", asymmetry)
assert asymmetry > 1e-10

||sigma - sigma^T|| = 0.014674367771594332


## 10. 习题 2.11：构造叉乘矩阵

$$
[\boldsymbol\omega]_\times\mathbf r
=
\boldsymbol\omega\times\mathbf r,
$$

其分量为

$$
([\boldsymbol\omega]_\times)_{ik}
=
\varepsilon_{ijk}\omega_j.
$$

In [19]:
def cross_matrix(omega_vec):
    omega_vec = np.asarray(omega_vec, dtype=float)
    X = np.zeros((3, 3))

    for i in range(3):
        for k in range(3):
            for j in range(3):
                X[i, k] += epsilon[i, j, k] * omega_vec[j]

    return X

omega = np.array([1.5, -2.0, 0.75])
r = np.array([0.4, 1.2, -0.6])

X = cross_matrix(omega)

Xr = np.zeros(3)
for i in range(3):
    for k in range(3):
        Xr[i] += X[i, k] * r[k]

cross_ref = np.cross(omega, r)

print("[omega]_x =")
print(X)
print("[omega]_x r =", Xr)
print("omega x r =", cross_ref)
print("antisymmetry residual norm =", np.linalg.norm(X + X.T))

assert np.allclose(Xr, cross_ref)
assert np.allclose(X.T, -X)

[omega]_x =
[[ 0.   -0.75 -2.  ]
 [ 0.75  0.   -1.5 ]
 [ 2.    1.5   0.  ]]
[omega]_x r = [0.3 1.2 2.6]
omega x r = [0.3 1.2 2.6]
antisymmetry residual norm = 0.0


## 11. 习题 2.13：对称张量的六分量压缩存储

采用顺序

$$
[11,22,33,12,23,13].
$$

这里只存储张量分量本身，不自动引入工程剪切量的因子 2。

In [20]:
pairs = [
    (0, 0),
    (1, 1),
    (2, 2),
    (0, 1),
    (1, 2),
    (0, 2),
]

def encode_symmetric(S):
    S = np.asarray(S, dtype=float)
    if S.shape != (3, 3):
        raise ValueError("S 必须是 3×3 数组。")
    if not np.allclose(S, S.T):
        raise ValueError("S 必须是对称张量。")

    packed = np.zeros(6)
    for K, (i, j) in enumerate(pairs):
        packed[K] = S[i, j]
    return packed

def decode_symmetric(packed):
    packed = np.asarray(packed, dtype=float)
    if packed.shape != (6,):
        raise ValueError("packed 必须含 6 个分量。")

    S = np.zeros((3, 3))
    for K, (i, j) in enumerate(pairs):
        S[i, j] = packed[K]
        S[j, i] = packed[K]
    return S

S1 = np.array([
    [4.0, 1.0, 2.0],
    [1.0, 5.0, 3.0],
    [2.0, 3.0, 6.0],
])

S2 = np.array([
    [2.0, -1.0, 0.5],
    [-1.0, 3.0, 1.5],
    [0.5, 1.5, 7.0],
])

p1 = encode_symmetric(S1)
p2 = encode_symmetric(S2)
S1_rec = decode_symmetric(p1)

print("packed S1 =", p1)
print("decoded S1 =")
print(S1_rec)

assert np.allclose(S1_rec, S1)

packed S1 = [4. 5. 6. 1. 3. 2.]
decoded S1 =
[[4. 1. 2.]
 [1. 5. 3.]
 [2. 3. 6.]]


完整 Frobenius 内积中，非对角分量各出现两次。因此六个原始压缩分量不能直接使用等权欧氏点积来代替 $\mathbf S_1:\mathbf S_2$。

In [21]:
frobenius = 0.0
for i in range(3):
    for j in range(3):
        frobenius += S1[i, j] * S2[i, j]

plain_packed_dot = 0.0
for K in range(6):
    plain_packed_dot += p1[K] * p2[K]

weighted_packed_dot = (
    p1[0] * p2[0]
    + p1[1] * p2[1]
    + p1[2] * p2[2]
    + 2.0 * p1[3] * p2[3]
    + 2.0 * p1[4] * p2[4]
    + 2.0 * p1[5] * p2[5]
)

print("Frobenius inner product =", frobenius)
print("plain packed dot =", plain_packed_dot)
print("weighted packed dot =", weighted_packed_dot)

assert not np.isclose(frobenius, plain_packed_dot)
assert np.isclose(frobenius, weighted_packed_dot)

Frobenius inner product = 74.0
plain packed dot = 69.5
weighted packed dot = 74.0


## 12. 习题 2.14：对流加速度

$$
u_1=ax_1x_2,
\qquad
u_2=bx_1^2,
\qquad
u_3=cx_3.
$$

计算

$$
a_i^{(\mathrm{conv})}
=
u_j\partial_j u_i.
$$

In [22]:
x1, x2, x3 = sp.symbols("x1 x2 x3")
a, b, c = sp.symbols("a b c")

coords = [x1, x2, x3]
u_sym = [
    a * x1 * x2,
    b * x1**2,
    c * x3,
]

aconv = [0, 0, 0]

for i in range(3):
    value = 0
    for j in range(3):
        value += u_sym[j] * sp.diff(u_sym[i], coords[j])
    aconv[i] = sp.simplify(value)

for i, expr in enumerate(aconv, start=1):
    print(f"a_conv_{i} =", expr)

a_conv_1 = a*x1*(a*x2**2 + b*x1**2)
a_conv_2 = 2*a*b*x1**2*x2
a_conv_3 = c**2*x3


结果为

$$
a_1^{(\mathrm{conv})}
=
a^2x_1x_2^2+abx_1^3,
$$

$$
a_2^{(\mathrm{conv})}
=
2abx_1^2x_2,
$$

$$
a_3^{(\mathrm{conv})}
=
c^2x_3.
$$

继续核对

$$
(\nabla\mathbf u)_{ij}=\partial_j u_i,
\qquad
\mathbf a^{(\mathrm{conv})}
=
(\nabla\mathbf u)\mathbf u.
$$

In [23]:
L = sp.MutableDenseMatrix(3, 3, [0] * 9)

for i in range(3):
    for j in range(3):
        L[i, j] = sp.diff(u_sym[i], coords[j])

aconv_matrix_form = [0, 0, 0]
for i in range(3):
    value = 0
    for j in range(3):
        value += L[i, j] * u_sym[j]
    aconv_matrix_form[i] = sp.simplify(value)

residuals = [
    sp.simplify(aconv[i] - aconv_matrix_form[i])
    for i in range(3)
]

print("velocity gradient L_ij = d_j u_i:")
sp.pprint(L)
print("residuals =", residuals)

assert residuals == [0, 0, 0]

velocity gradient L_ij = d_j u_i:
⎡ a⋅x₂   a⋅x₁  0⎤
⎢               ⎥
⎢2⋅b⋅x₁   0    0⎥
⎢               ⎥
⎣  0      0    c⎦
residuals = [0, 0, 0]


## 13. 习题 2.15：指标表达式语法检查器原型

完整 LaTeX 解析器需要词法分析、语法树、作用域和算子优先级。这里仅做一个教学型简化原型。

输入格式示例：

```text
A_ij*B_jk + C_ik
```

In [24]:
index_pattern = re.compile(r"_([a-zA-Z]+)")

def term_index_counts(term):
    groups = index_pattern.findall(term)
    counts = {}
    for group in groups:
        for idx in group:
            counts[idx] = counts.get(idx, 0) + 1
    return counts

def audit_term(term):
    counts = term_index_counts(term)

    illegal = {idx: n for idx, n in counts.items() if n >= 3}
    free = sorted([idx for idx, n in counts.items() if n == 1])
    dummy = sorted([idx for idx, n in counts.items() if n == 2])

    return {
        "term": term.strip(),
        "counts": counts,
        "free": free,
        "dummy": dummy,
        "illegal": illegal,
    }

def audit_sum_expression(expr):
    parts = re.split(r"\+|-", expr)
    parts = [p.strip() for p in parts if p.strip()]

    reports = [audit_term(p) for p in parts]
    free_sets = [tuple(r["free"]) for r in reports]

    same_free_indices = len(set(free_sets)) <= 1
    no_illegal_repeats = all(not r["illegal"] for r in reports)

    return {
        "expression": expr,
        "terms": reports,
        "same_free_indices": same_free_indices,
        "no_illegal_repeats": no_illegal_repeats,
    }

tests = [
    "A_ij*B_jk + C_ik",
    "A_ii*x_i",
    "A_ij*B_jk*x_k + C_im",
    "epsilon_ijk*u_j*v_k",
    "delta_ij*A_jk + B_ik",
    "A_ij*B_kl + C_ik*D_jl",
]

for expr in tests:
    report = audit_sum_expression(expr)
    print("\n", expr)
    print("same free indices:", report["same_free_indices"])
    print("no illegal repeats:", report["no_illegal_repeats"])
    for term_report in report["terms"]:
        print("  ", term_report)


 A_ij*B_jk + C_ik
same free indices: True
no illegal repeats: True
   {'term': 'A_ij*B_jk', 'counts': {'i': 1, 'j': 2, 'k': 1}, 'free': ['i', 'k'], 'dummy': ['j'], 'illegal': {}}
   {'term': 'C_ik', 'counts': {'i': 1, 'k': 1}, 'free': ['i', 'k'], 'dummy': [], 'illegal': {}}

 A_ii*x_i
same free indices: True
no illegal repeats: False
   {'term': 'A_ii*x_i', 'counts': {'i': 3}, 'free': [], 'dummy': [], 'illegal': {'i': 3}}

 A_ij*B_jk*x_k + C_im
same free indices: False
no illegal repeats: True
   {'term': 'A_ij*B_jk*x_k', 'counts': {'i': 1, 'j': 2, 'k': 2}, 'free': ['i'], 'dummy': ['j', 'k'], 'illegal': {}}
   {'term': 'C_im', 'counts': {'i': 1, 'm': 1}, 'free': ['i', 'm'], 'dummy': [], 'illegal': {}}

 epsilon_ijk*u_j*v_k
same free indices: True
no illegal repeats: True
   {'term': 'epsilon_ijk*u_j*v_k', 'counts': {'i': 1, 'j': 2, 'k': 2}, 'free': ['i'], 'dummy': ['j', 'k'], 'illegal': {}}

 delta_ij*A_jk + B_ik
same free indices: True
no illegal repeats: True
   {'term': 'delta_ij*A

### 原型的能力边界

仅靠指标计数可以发现：

1. 同一乘积项某指标出现三次或更多；
2. 同一加减式各项自由指标集合不一致；
3. 若分别审计等式左右两端，也可以比较两端自由指标集合。

仅靠指标计数不能可靠判断：

- $\partial_i(\phi v_i)$ 与 $\phi\,\partial_i v_i$ 的作用范围差异；
- 一个符号究竟声明为几阶张量；
- 两个本应独立的哑指标被误改为同一字母、但碰巧仍满足计数规则的语义错误；
- 公式的单位、物理定义和本构假设；
- 复杂括号、函数、导数与复合算子的完整语法。

真正的科学计算语法检查器需要建立抽象语法树（AST），而不是只做正则表达式计数。

## 14. 本章计算关系总表

$$
\boxed{
y_i=A_{ij}u_j
}
$$

$$
\boxed{
C_{ik}=A_{ij}B_{jk}
}
$$

$$
\boxed{
\mathbf A:\mathbf B=A_{ij}B_{ij}
}
$$

$$
\boxed{
\delta_{ij}u_j=u_i
}
$$

$$
\boxed{
w_i=\varepsilon_{ijk}u_jv_k
}
$$

$$
\boxed{
\varepsilon_{ijk}\varepsilon_{\ell jk}
=
2\delta_{i\ell}
}
$$

$$
\boxed{
\sigma_{ij}=C_{ijkl}\varepsilon_{kl}
}
$$

最重要的程序翻译原则：

- **自由指标 $\rightarrow$ 输出位置 / 外层循环；**
- **哑指标 $\rightarrow$ 内部求和 / 内层循环；**
- **无自由指标 $\rightarrow$ 最终输出为标量。**

### 建议进一步探索

1. 为语法检查器增加等号两侧自由指标比较；
2. 为简化指标字符串建立 token 和抽象语法树；
3. 比较不同六分量存储顺序的数据布局；
4. 将六分量张量存储与工程剪切应变因子 2 结合；
5. 用随机矩阵系统比较 $A_{ij}A_{ij}$ 与 $A_{ij}A_{ji}$；
6. 在第三章建立坐标变换后继续研究 $\varepsilon_{ijk}$ 的空间取向性质。